# Jeddah flood dataset — data preparation

This notebook builds the whole dataset step by step, for **both storms** (21 Nov 2017 and 24 Nov 2022 — listed under `events:` in `config.yaml`). Run the cells from top to bottom.

**Version 2:** two storms instead of one; the 2022 rows are identical to version 1.

**Before you start (only once):**
1. In a terminal, inside the project folder, run:
   `python -m venv .venv` and then `.venv\Scripts\python -m pip install -r requirements.txt`
2. Connect this notebook to that environment:
   `.venv\Scripts\python -m ipykernel install --sys-prefix --name saudi-water-paradox --display-name "Python (saudi-water-paradox .venv)"`
   Then pick **Python (saudi-water-paradox .venv)** as the kernel.
3. Make a free **NASA Earthdata** account (needed for the rain and soil steps).

**Result:** the two tables in `data/processed/` (both storms in each table), the column guide `data_dictionary.csv`, and the check report `docs/qa_report.md`.

## Setup
Finds the project folder and prepares a helper that runs each step.

In [ ]:
import hashlib, os, re, subprocess, sys, time
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "config.yaml").exists() else Path.cwd().parent
assert (ROOT / "config.yaml").exists(), "Open this notebook from the project folder or from notebooks/"
PY = sys.executable  # the kernel's Python = the project .venv

try:
    import numpy, pandas, rasterio, pysheds, earthaccess  # noqa: F401
except ImportError as e:
    raise SystemExit(f"Missing package ({e.name}). Select the kernel 'Python (saudi-water-paradox .venv)'.")

def _redact(text: str) -> str:
    """Hide local folder paths so the saved notebook contains nothing personal."""
    text = text.replace(str(ROOT), "<project>").replace(str(Path.home()), "~")
    return re.sub(r"[A-Za-z]:\\Users\\[^\\\s]+", "<user-home>", text)

def run(script: str) -> None:
    t = time.time()
    print(f"=== {script}")
    r = subprocess.run([PY, str(ROOT / "scripts" / script)], cwd=ROOT, capture_output=True, text=True,
                       encoding="utf-8", errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    out = _redact(r.stdout)
    print(out if len(out) < 3000 else "...\n" + out[-3000:])
    if r.returncode != 0:
        print(_redact(r.stderr)[-3000:])
        raise RuntimeError(f"{script} failed (exit code {r.returncode})")
    print(f"--- done in {time.time() - t:.0f} s")

print("project folder found; Python", sys.version.split()[0])

## Step 1 — Check the flood maps
For each storm, compares the satellite flood map with normal photos from before and after the storm.
Saves maps to `docs/checks/<event_id>/` for people to look at. Needs internet.

In [ ]:
run("01_probe_labels.py")

## Step 2 — Download the height map
Downloads the land height map for Jeddah and the valleys around it (about 225 MB). Needs internet.

In [ ]:
run("02_download_dem.py")

## Step 3 — Download the land-cover map
Downloads the map that shows buildings, bare land, plants and water (about 26 MB). Needs internet.

In [ ]:
run("03_download_worldcover.py")

## Step 4 — Check your NASA login
The next two steps download data from NASA, so you must be logged in.
Do this once in a terminal (you type your own password; it is not saved in the project):

`.venv\Scripts\python -c "import earthaccess; earthaccess.login(strategy='interactive', persist=True)"`

Then on the Earthdata website go to *Profile → Applications → Authorized Apps* and approve **NASA GESDISC DATA ARCHIVE**.
This cell only checks that the login works.

In [ ]:
check = "import earthaccess, sys; a = earthaccess.login(strategy='netrc'); sys.exit(0 if a and a.authenticated else 1)"
r = subprocess.run([PY, "-c", check], capture_output=True, text=True)
if r.returncode == 0:
    print("NASA Earthdata login: OK")
else:
    raise SystemExit("NASA Earthdata login NOT found or not working: run the terminal command above, then re-run this cell.")

## Step 5 — Download rain data
Downloads rain every 30 minutes, for the Jeddah area only, from 7 days before each storm until the day of
its radar image: 14–25 Nov 2017 and 17–29 Nov 2022 (about 44 MB in total). Needs the NASA login.

In [ ]:
run("04_download_imerg.py")

## Step 6 — Download soil moisture
Downloads how wet the soil was just before each storm started. Needs the NASA login.

In [ ]:
run("05_download_smap.py")

## Step 7 — Download the flood-map files
Saves the satellite flood maps (before and after each storm) and the photos used to check them
(about 395 MB). Needs internet.

In [ ]:
run("06_download_labels.py")

## Step 8 — Work out the rain numbers
For each storm: rain on flood day, the heaviest half hour, and rain in the week before.
Also checks that no rain fell between the storm and the satellite image.

In [ ]:
run("07_derive_rain.py")

## Step 9 — Work out the land and water-flow numbers
Finds the valleys that drain into Jeddah, then calculates slope, where water flows and collects,
and how high each place is above the nearest stream or the sea. This is the slowest step (a few minutes).

In [ ]:
run("08_derive_terrain.py")

## Step 10 — Build the tables
Puts everything together, marks each square as flooded (1) or dry (0) for each storm, and saves the two tables.
Water already seen on the radar image *before* a storm is left out (neither 1 nor 0).

In [ ]:
run("09_build_table.py")

## Step 11 — Quality checks
Runs automatic checks on both tables, for the whole file and for each storm, and writes `docs/qa_report.md`.
Stops with an error if any check fails.

In [ ]:
run("10_qa.py")

## Final check
Shows the row counts and the file fingerprint (SHA-256, a code that changes if the file changes).
It should match the one in `docs/qa_report.md`.

In [ ]:
import pandas as pd

for name in ("jeddah_flood_ml", "jeddah_flood_ml_elev_matched"):
    sha = hashlib.sha256((ROOT / "data/processed" / f"{name}.csv").read_bytes()).hexdigest()
    df = pd.read_parquet(ROOT / "data/processed" / f"{name}.parquet")
    counts = df.label.value_counts().to_dict()
    print(f"{name}: {len(df):,} rows | flooded (1): {counts.get(1, 0):,} | dry (0): {counts.get(0, 0):,}")
    for ev, d in df.groupby("event_id"):
        c = d.label.value_counts().to_dict()
        print(f"  {ev}: {len(d):,} rows | flooded (1): {c.get(1, 0):,} | dry (0): {c.get(0, 0):,}")
    print(f"  SHA-256 {sha}")

dd = pd.read_csv(ROOT / "data_dictionary.csv")
print("\nColumns allowed for training:", ", ".join(dd.loc[dd.use_as_model_input == "yes", "column"]))